In [ ]:
!pip install unsloth

In [ ]:
%load_ext tensorboard
%tensorboard --logdir local_output_dir

In [ ]:
from datasets import load_dataset
xlam_fc = load_dataset("Salesforce/xlam-function-calling-60k", split = "train")
xlam_irr = load_dataset("MadeAgents/xlam-irrelevance-7.5k", split="train")

In [ ]:
xlam_fc

In [ ]:
xlam_irr

In [ ]:
from datasets import load_dataset, concatenate_datasets

# Remove 'id' to make the schemas identical
xlam_fc = xlam_fc.remove_columns(["id"])

# Combine and shuffle
dataset = concatenate_datasets([xlam_fc, xlam_irr])
dataset = dataset.shuffle(seed=42)

In [ ]:
dataset

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:

import os

# Define where to save inside Google Drive
drive_output_dir = "/content/drive/MyDrive/colab_checkpoints/qwen3-1.7_expert_tools_v2"

# Local output dir (Colab temporary storage)
local_output_dir = "outputs"

# Make sure dirs exist
os.makedirs(drive_output_dir, exist_ok=True)
os.makedirs(local_output_dir, exist_ok=True)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen3-1.7B",
    load_in_4bit = True,
    load_in_8bit = False,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    r = 64,
    lora_alpha = 128,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

In [ ]:
dataset[65000]

{'query': "Retrieve box office information for the movie with identifier 'tt1234567' and fetch the latest NFT news related to 'crypto art'.",
 'answers': '[{"name": "title_v2_get_business", "arguments": {"tconst": "tt1234567"}}, {"name": "nft_news", "arguments": {"nftnews": "crypto art"}}]',
 'tools': '[{"name": "nft_news", "description": "Fetches up-to-date NFT news from the specified API.", "parameters": {"nftnews": {"description": "A specific filter to narrow down NFT news. Default is None, meaning no specific filter.", "type": "str, optional", "default": ""}}}, {"name": "theaters_list", "description": "List theaters around a specified postal code or GEO location using the Flixster API.", "parameters": {"longitude": {"description": "The GEO longitude to search for theaters.", "type": "int, optional", "default": ""}, "zipcode": {"description": "The postal code to search for theaters. Default is \'90002\'.", "type": "str, optional", "default": "90002"}, "latitude": {"description": "Th

In [ ]:
# original
from unsloth.chat_templates import get_chat_template

# Initialize the tokenizer with the correct Qwen3 template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-3", # Use the built-in unsloth qwen-3 template
)

def formatting_prompts_func(sample):
    import json

    tools = json.loads(sample["tools"])
    ans_list = json.loads(sample["answers"])

    tool_calls = []
    for a in ans_list:
        tool_calls.append({
            "type": "function",
            "function": {
                "name": a["name"],
                "arguments": json.dumps(a["arguments"])  # <-- must be a string, not dict
            }
        })

    messages = [
        {"role": "user", "content": sample["query"]},
        {
            "role": "assistant",
            "content": "",
            "tool_calls": tool_calls
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text}

dataset = dataset.map(formatting_prompts_func, batched=False)

Map:   0%|          | 0/67500 [00:00<?, ? examples/s]

In [ ]:
print(dataset[65000]["text"])

NameError: name 'dataset' is not defined

In [ ]:
dataset["text"][1]

'<|im_start|>system\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>\n{"name": "search_by_name", "description": "Searches for a movie by its title using the provided query string.", "parameters": {"query": {"description": "The movie title or keyword to search for.", "type": "str", "default": "kong"}, "page": {"description": "The page number of results to retrieve. Defaults to 1.", "type": "int, optional", "default": "1"}}}\n{"name": "search_by_genre", "description": "Discover movies by genre using the RapidAPI.", "parameters": {"with_genres": {"description": "The genre code to filter movies by. Defaults to \'80\'.", "type": "str", "default": "80"}, "page": {"description": "The page number of results to retrieve. Defaults to 1.", "type": "int", "default": "1"}}}\n{"name": "get_video", "description": "Fetches video data using a query string from the Playphrase API.", "param

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        # warmup_ratio = 0.05, #is deprecated and will be removed in v5.2
        # max_steps = 200,
        num_train_epochs = 1,
        learning_rate = 5e-6,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        # output_dir = "outputs",
        remove_unused_columns = True,
        packing = False,
        save_steps=25,
        # save_steps=100,
        save_total_limit=2,
        output_dir=local_output_dir,
        # report_to = "none",
        report_to = "tensorboard",
        # logging_dir = "./logs", #is deprecated and will be removed in v5.2
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/67500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
trainer_stats = trainer.train()
# trainer_stats = trainer.train(resume_from_checkpoint=drive_output_dir + "/checkpoint-950") # if needed to use stored chackpoints.

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 67,500 | Num Epochs = 1 | Total steps = 2,110
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 69,730,304 of 1,790,305,280 (3.89% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
951,0.528416
952,0.567978
953,0.445211
954,0.482458
955,0.620250
956,0.483541
957,0.544598
958,0.503300
959,0.465560
960,0.401440


In [ ]:
# To save checkpoint on drive
import shutil
if os.path.exists(local_output_dir):
  shutil.copytree(local_output_dir, drive_output_dir, dirs_exist_ok=True)
  print("Checkpoints saved to:", drive_output_dir)

In [ ]:
# del model.config.quantization_config # enabla this if this shows an error about model config files. deleting this do not affect anything.
if True: model.push_to_hub_merged("MadhuryaPasan/qwen3-1.7_expert_tools_v2", tokenizer, token = "hf_xxx")

config.json:   0%|          | 0.00/752 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tools_v0_1/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ls_v0_1/model.safetensors:   0%|          | 16.7MB / 3.44GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:01<00:00, 121.23s/it]


Unsloth: Merge process complete. Saved to `/content/MadhuryaPasan/qwen3_expert_tools_v0_1`


In [ ]:
# del model.config.quantization_config # enabla this if this shows an error about model config files. deleting this do not affect anything.

if True:
    model.push_to_hub_gguf(
        "MadhuryaPasan/qwen3-1.7_expert_tools_v2_gguf",
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0"],
        token = "hf_xxx",
    )

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:55<00:00, 55.03s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


...

Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/tmp/unsloth_gguf_b804ui8c_gguf/qwen3-1.7b.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/tmp/unsloth_gguf_b804ui8c_gguf/qwen3-1.7b.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: llama.cpp/llama-cli --model /tmp/unsloth_gguf_b804ui8c_gguf/qwen3-1.7b.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /tmp/unsloth_gguf_b804ui8c_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /tmp/unsloth_gguf_b804ui8c_gguf/Modelfile
Unsloth: Uploading GGUF to Huggingface Hub...
Uploading qwen3-1.7b.Q4_K_M.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uf/qwen3-1.7b.Q4_K_M.gguf:   0%|          |  548kB / 1.11GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/MadhuryaPasan/qwen3_expert_tools_v0_1_gguf
Unsloth: Cleaning up temporary files...


In [ ]:
# Save only the LoRA adapters
model.save_pretrained("qwen3-1.7_expert_tools_v2_lora_adapters")
tokenizer.save_pretrained("qwen3-1.7_expert_tools_v2_lora_adapters")

[]

In [ ]:
# To automatically end the colab session

from google.colab import runtime
runtime.unassign()